In [ ]:
from dotenv import load_dotenv
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_classic.prompts import ChatPromptTemplate
from langchain_classic.storage import LocalFileStore
from langchain_classic.vectorstores import FAISS
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_unstructured import UnstructuredLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores.utils import filter_complex_metadata

load_dotenv()

llm = ChatOpenAI(
  temperature=0.1,
)

# 1. load and transform documents
splitter = CharacterTextSplitter.from_tiktoken_encoder(
  separator="\n",
  chunk_size=600, # 600자 청크로 분할
  chunk_overlap=100, # 청크 간 100자 겹침으로 문맥 유지
)

loader = UnstructuredLoader("./files/How_Neflix_Uses_Java_-_2026_.pdf")

docs = filter_complex_metadata(loader.load_and_split(text_splitter=splitter))


# 2. embeddings with caching
cache_dir = LocalFileStore("./.cache/")

embeddings = OpenAIEmbeddings()

cached_embeddings = CacheBackedEmbeddings.from_bytes_store(embeddings, cache_dir)


# 3. store in vector database
vectorstore = FAISS.from_documents(docs, cached_embeddings)

# 임베딩 작업을 할 때 먼저 캐시에 embeddings가 있는지 확인하고, 없으면 OpenAI API를 호출하여 임베딩을 생성한 후 캐시에 저장한다. 
# 이렇게 하면 동일한 텍스트에 대해 여러 번 임베딩을 생성할 때 API 호출을 줄일 수 있다.


# 4. create retriever and Map Reduce LCEL chain
retriever = vectorstore.as_retriever()

# 4-1. Map 단계: 각 문서 청크에 대해 질문과 함께 LLM을 호출하여 관련 텍스트를 추출한다.
map_doc_prompt = ChatPromptTemplate.from_messages([
  (
    "system",
    """
    Use the following portion of a long document to see if any of the text is relevant to answer the question. Return any relevant text verbatim.
    ------
    {context}
    """
  ),
  ("human", "{question}")
])

map_doc_chain = map_doc_prompt | llm

def map_docs(inputs):
  documents = inputs["documents"]
  question = inputs["question"]
  
  return "\n\n".join(map_doc_chain.invoke({
      "context": doc.page_content,
      "question": question,
    }).content for doc in documents
  )

map_chain = {
  "documents": retriever, 
  "question": RunnablePassthrough()
} | RunnableLambda(map_docs)

# 4-2. Reduce 단계: Map 단계에서 추출된 관련 텍스트를 바탕으로 최종 답변을 생성한다.
final_prompt = ChatPromptTemplate.from_messages([
  (
    "system", 
    """
    Given the following extracted parts of a long document and a question, create a final answer.
    If you don't know the answer, just say that you don't know. Don't try to make up an answer.
    ------
    {context}
    """
  ),
  ("human", "{question}")
])

final_chain = {"context": map_chain, "question": RunnablePassthrough()} | final_prompt | llm

# 현재 RAG 검색형 질문을 위한 구조라서 문서 전체 주제보다는 문서 안에 있을 법한 구체적인 정보를 묻는 것이 좋다.
query = "What are the main reasons Netflix continues to use Java?"

final_chain.invoke(query)

INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


46
